In [1]:
import socket
import time

# การตั้งค่า IP และ Port ให้ตรงกับ TM-X
TMX_IP = '192.168.10.11'
TMX_PORT = 8600
BUFFER_SIZE = 1024
values = []
def send_command(sock, command):
    """
    ฟังก์ชันสำหรับส่งคำสั่งไปยัง TM-X และรอรับผลลัพธ์ตอบกลับ
    """
    # ต้องต่อท้ายด้วยตัวคั่น (Delimiter) เสมอ ในที่นี้คือ CR (\r)
    cmd_to_send = command + '\r'
    
    # ส่งข้อมูลไปยังกล้อง
    sock.sendall(cmd_to_send.encode('ascii'))
    time.sleep(0.1) # หน่วงเวลาให้กล้องประมวลผลเล็กน้อย
    
    try:
        # รับข้อความตอบกลับจากกล้อง
        response = sock.recv(BUFFER_SIZE).decode('ascii').strip()
        # print(f"[ส่งคำสั่ง]: {command.ljust(15)} | [ตอบกลับ]: {response}")
        return response
    except Exception as e:
        print(f"Error reading response: {e}")
        return None

def main():
    # สร้าง TCP Socket
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client_socket.settimeout(5.0) # กำหนด Timeout 5 วินาที
    
    try:
        print(f"Connecting to TM-X at {TMX_IP}:{TMX_PORT}...")
        client_socket.connect((TMX_IP, TMX_PORT))
        print("Connected successfully!\n")
        
        # ----------------------------------------------------
        # ตัวอย่างการสั่งงานเบื้องต้น
        # ----------------------------------------------------
        
        # 1. เข้าสู่โหมด Run
        send_command(client_socket, "R0")
        time.sleep(0.5)
        
        # 2. เปลี่ยนโปรแกรม
        send_command(client_socket, "PW,1,021")
        print("Waiting for program to load...")
        time.sleep(1.0)  # <-- เพิ่มเวลาตรงนี้ให้กล้องเตรียมตัว
        
        # 3. ล้าง Error (ย้ายมาไว้ก่อนทริกเกอร์)
        
        # 4. ส่งคำสั่งทริกเกอร์
        while(1):
            response_data = send_command(client_socket, "GM,0,0").split(',')
            # time.sleep(0.5)


            for i in response_data:
                i = i.strip('-')
                i = i.strip('+')
                # print(i)

                if i == "9999.999":
                    pass
                else:
                    # val = float(i[3:])
                    val = i
                    values.append(val)
        # send_command(client_socket, "GR,0")
        # time.sleep(0.5)

            print(f"Values : {values[-2:]}\n")
        

    except Exception as e:
        print(f"Connection failed or error occurred: {e}")
    finally:
        # ปิดการเชื่อมต่อเสมอเมื่อใช้งานเสร็จ
        client_socket.close()
        print("\nConnection closed.")

if __name__ == "__main__":
    main()

Connecting to TM-X at 192.168.10.11:8600...
Connection failed or error occurred: timed out

Connection closed.


In [ ]:
from pyftpdlib.authorizers import DummyAuthorizer
from pyftpdlib.handlers import FTPHandler
from pyftpdlib.servers import FTPServer

def start_ftp():
    authorizer = DummyAuthorizer()
    
    # กำหนด Username, Password, โฟลเดอร์ที่ใช้รับไฟล์ และสิทธิ์ 'elradfmw' (อ่าน/เขียน/แก้ไขได้เต็มที่)
    authorizer.add_user("INTERN_USER", "123456", "D:\\MatchaLatte\\TM-X Improvement\\images", perm="elradfmw")
    
    handler = FTPHandler
    handler.authorizer = authorizer
    
    # ระบุ IP Address ของ PC คุณ (0.0.0.0 หมายถึงรับทุก IP ในเครื่อง)
    # พอร์ตมาตรฐานของ FTP คือ 21|
    server = FTPServer(("0.0.0.0", 21), handler)
    print("FTP Server is running... Waiting for Keyence images.")
    server.serve_forever()

if __name__ == "__main__":
    start_ftp()

[I 2026-07-22 15:05:59] concurrency model: async
[I 2026-07-22 15:05:59] masquerade (NAT) address: None
[I 2026-07-22 15:05:59] passive ports: None
[I 2026-07-22 15:05:59] >>> starting FTP server on 0.0.0.0:21, pid=32948 <<<


FTP Server is running... Waiting for Keyence images.


In [ ]:
import os
import threading
from pyftpdlib.authorizers import DummyAuthorizer
from pyftpdlib.handlers import FTPHandler
from pyftpdlib.servers import FTPServer

# ตัวแปร Flag สำหรับควบคุมการบันทึกรูป
accept_next_image = False

class FilteredFTPHandler(FTPHandler):
    def on_file_received(self, file):
        global accept_next_image
        
        # ถ้ารอรับรูปอยู่ (กด Enter มาแล้ว)
        if accept_next_image:
            print(f"\n[+] Saved successfully: {os.path.basename(file)}")
            # รับแล้วรีเซ็ต Flag กลับเพื่อรอกด Enter ครั้งต่อไป
            accept_next_image = False 
            print("\nPress Enter to capture 1 image...", end="", flush=True)
        else:
            # ถ้าไม่ได้กด Enter รูปที่ Controller ส่งมาจะถูกลบทิ้งทันที
            try:
                os.remove(file)
            except OSError:
                pass

def start_ftp():
    authorizer = DummyAuthorizer()
    authorizer.add_user("INTERN_USER", "123456", "C:\\Users\\PMehom\\Desktop\\TM-X\\Images", perm="elradfmw")
    
    # ใช้ Handler ที่เรา Custom ขึ้นมา
    handler = FilteredFTPHandler
    handler.authorizer = authorizer
    
    server = FTPServer(("0.0.0.0", 21), handler)
    server.serve_forever()

if __name__ == "__main__":
    # 1. รัน FTP Server ใน Thread ย่อยเบื้องหลัง เพื่อไม่ให้บล็อกการทำงาน
    ftp_thread = threading.Thread(target=start_ftp, daemon=True)
    ftp_thread.start()
    
    print("FTP Server is running in background...")
    print("Waiting for Keyence controller...")
    
    # 2. Main Thread วนลูปเพื่อรอรับการกด Enter
    while True:
        input("\nPress Enter to capture 1 image...")
        accept_next_image = True

Exception in thread Thread-5 (start_ftp):
Traceback (most recent call last):
  File "c:\Users\Lenovo Legion\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner


FTP Server is running in background...
Waiting for Keyence controller...


In [ ]:
import threading
import socket
from pyftpdlib.authorizers import DummyAuthorizer
from pyftpdlib.handlers import FTPHandler
from pyftpdlib.servers import FTPServer

# ---------------------------------------------------------
# ส่วนที่ 1: ตั้งค่า FTP Server (ทำงานเบื้องหลัง)
# ---------------------------------------------------------
def start_ftp():
    authorizer = DummyAuthorizer()
    authorizer.add_user("INTERN_USER", "123456", "C:\\Users\\PMehom\\Desktop\\TM-X\\Images", perm="elradfmw")
    
    handler = FTPHandler
    handler.authorizer = authorizer
    
    server = FTPServer(("0.0.0.0", 21), handler)
    server.serve_forever()

# ---------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชันส่งคำสั่ง Trigger ไปที่ Keyence Controller
# ---------------------------------------------------------
def trigger_keyence(controller_ip, port=8500):
    try:
        # สร้างการเชื่อมต่อ TCP/IP ไปที่ Controller
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(2.0)
        s.connect((controller_ip, port))
        
        # คำสั่ง Trigger (*** คุณต้องเช็คคำสั่งที่ถูกต้องจากคู่มือ TM-X อีกครั้ง ***)
        # ตัวอย่างเช่น "T1", "M1" หรือตามด้วย Carriage Return (\r)
        command = "T1\r" 
        s.send(command.encode('ascii'))
        
        s.close()
        print("[+] Trigger sent to Keyence successfully!")
    except Exception as e:
        print(f"[-] Error sending trigger: {e}")

# ---------------------------------------------------------
# ส่วนที่ 3: Main Program (วนลูปรอกด Enter)
# ---------------------------------------------------------
if __name__ == "__main__":
    # รัน FTP Server ใน Thread แยกต่างหาก
    ftp_thread = threading.Thread(target=start_ftp, daemon=True)
    ftp_thread.start()
    
    print("FTP Server is running in background...")
    
    # ใส่ IP Address ของ Keyence TM-X065 ของคุณ
    KEYENCE_IP = "192.168.0.10" 
    # พอร์ตรับคำสั่งของ Keyence (มักจะเป็น 8500 หรือพอร์ตที่คุณตั้งไว้)
    KEYENCE_PORT = 8600 

    while True:
        # ระบบจะหยุดรอจนกว่าคุณจะกด Enter
        input("\nPress Enter to capture 1 image...")
        
        # เมื่อกด Enter จะส่งคำสั่งไปกระตุ้น 
        # Controller
        trigger_keyence(KEYENCE_IP, KEYENCE_PORT)
        
        # หลังจากส่งคำสั่ง Controller จะถ่ายรูป 1 ครั้ง
        # และส่งรูปนั้นกลับมาที่ FTP Server ของเราเองโดยอัตโนมัติ

Exception in thread Thread-5 (start_ftp):
Traceback (most recent call last):
  File "c:\Users\Lenovo Legion\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner


FTP Server is running in background...
